# Does knowing the borough improve my pickup forecast?

My [forecasting model](04_production_modeling_refactored_v2.ipynb) already knows the taxi zone. So why give it the borough too? Two zones in Queens have different IDs, but the borough label gives a tree one way to group them. I want to see whether making that grouping easier helps it predict completed pickups.

I train two XGBoost models from scratch. One gets the original 30 features and the other gets those same features plus borough. Both get 1,613 rounds and see the same training and validation rows. I leave the later test period closed because this validation comparison is still part of deciding what to keep.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import sys
import time
import uuid
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import xgboost as xgb
from IPython.display import display

def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from within nyc-ride-demand-platform.')

ROOT = find_project_root(Path.cwd())
DATA = ROOT / 'data/gold/production_model/hourly_zone_horizon_features/availability_delay=7h'
BUNDLE = ROOT / 'models/evaluation/xgboost_263_zone_evaluation_2026-08-25'
LOOKUP = ROOT / 'data/external/taxi_zone_lookup.csv'
DEVICE = 'cuda'  # Change to 'cpu' if needed; CPU training may be much slower.
SEEDS = [42]  # For a stability check, set [42, 43, 44] before running both arms.
ROUNDS = 1613
RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = ROOT / 'models/experiments/borough_feature' / RUN_ID
print('Project:', ROOT)
print('XGBoost:', xgb.__version__, '| device:', DEVICE)
print('Outputs will be isolated in:', OUT)


## What can I reuse from the earlier work?

I check the feature tables and the saved model configuration first. I can reuse the settings, but the saved model was trained with rows I'm about to use for validation. If I scored those fitted trees here, I'd give them credit for predicting outcomes they had already seen.

The saved wait-time table is a separate input for the [pickup wait baseline](09_wait_time_baseline.ipynb). It contains averages. To forecast a median wait, I would need to build that target from trip-level records.


In [ ]:
asset_rows = []
for relative in [
    'data/gold/zone_hour_pickups.parquet',
    'data/gold/zone_hour_pickup_lags.parquet',
    'data/gold/zone_hour_wait_time.parquet',
]:
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(path)
    file = pq.ParquetFile(path)
    asset_rows.append({'asset': relative, 'rows': file.metadata.num_rows,
                       'size_mb': round(path.stat().st_size / 1024**2, 2),
                       'columns': file.schema_arrow.names})
display(pd.DataFrame(asset_rows))


## How do I match taxi zones to boroughs?

I use the [TLC taxi zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) and check that each zone maps to exactly one category. I keep Newark Airport as its own category because it was in the earlier 263-zone evaluation. Both models still predict the same zones.


In [ ]:
lookup = pd.read_csv(LOOKUP, usecols=['LocationID', 'Borough'])
lookup = lookup.rename(columns={'LocationID': 'location_id', 'Borough': 'borough'})
lookup = lookup.loc[lookup.location_id.between(1, 263)].copy()
assert lookup.location_id.is_unique, 'Duplicate zone mappings must be resolved.'
assert set(lookup.location_id) == set(range(1, 264))
assert lookup.borough.notna().all()
lookup.borough = lookup.borough.str.strip()
assert lookup.borough.ne('').all()
assert set(lookup.borough) == {'Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island', 'EWR'}
borough_map = lookup.set_index('location_id').borough
borough_levels = sorted(lookup.borough.unique())
display(lookup.groupby('borough').size().rename('zones').to_frame())

manifest = pd.read_parquet(DATA / 'manifests/production_model_feature_manifest.parquet')
features = manifest.loc[manifest.use_in_primary_model, 'column_name'].tolist()
contract = json.loads((BUNDLE / 'feature_contract.json').read_text())
assert features == contract['ordered_feature_names']
assert len(features) == 30 and 'borough' not in features and 'total_pickups' not in features
category_levels = {
    'location_id': list(range(1, 264)), 'forecast_horizon': list(range(1, 7)),
    'hour_of_day': list(range(24)), 'day_of_week': list(range(7)),
    'month': list(range(1, 13)),
}
assert set(manifest.loc[manifest.use_in_primary_model & manifest.modeling_type.eq('categorical'),
                        'column_name']) == set(category_levels)


## Keeping the comparison on the same dates

I use the saved time split so both models train on the same earlier forecasts and predict the same later validation hours. I check that each origin has all six horizons and 263 zones, and that no target hour crosses from training into validation. Loading the full tables takes substantial memory.


In [ ]:
splits = pd.read_parquet(DATA / 'manifests/production_forecast_origin_split_manifest.parquet')
assert splits.forecast_origin_utc.is_unique
train_origins = splits.loc[splits.split.eq('training')]
valid_origins = splits.loc[splits.split.eq('validation')]
assert len(train_origins) and len(valid_origins)
assert train_origins.target_end_utc.max() < valid_origins.target_start_utc.min()
paths = sorted(DATA.glob('year=*/month=*/production_features_*.parquet'))
assert paths
dataset = ds.dataset([str(p) for p in paths], format='parquet')
keys = ['forecast_origin_utc', 'target_hour_utc', 'location_id', 'forecast_horizon']
columns = list(dict.fromkeys(keys + features + ['total_pickups']))

def load_split(origins):
    table = dataset.to_table(columns=columns,
        filter=ds.field('forecast_origin_utc').isin(origins.forecast_origin_utc.tolist()))
    frame = table.to_pandas()
    del table
    assert len(frame) == int(origins.rows.sum())
    assert set(frame.forecast_origin_utc.unique()) == set(origins.forecast_origin_utc.unique())
    assert not frame.duplicated(['forecast_origin_utc', 'location_id', 'forecast_horizon']).any()
    assert frame.groupby('forecast_origin_utc').size().eq(263 * 6).all()
    assert frame.total_pickups.notna().all() and frame.total_pickups.ge(0).all()
    assert set(frame.forecast_horizon.unique()) == set(range(1, 7))
    frame = frame.sort_values(['forecast_origin_utc', 'forecast_horizon', 'location_id']).reset_index(drop=True)
    frame['borough'] = pd.Categorical(frame.location_id.map(borough_map), categories=borough_levels)
    assert frame.borough.notna().all()
    for name, levels in category_levels.items():
        assert frame[name].isin(levels).all(), name
        frame[name] = pd.Categorical(frame[name], categories=levels)
    return frame

training = load_split(train_origins)
validation = load_split(valid_origins)
print('Training:', training.shape, '| Validation:', validation.shape)
print('Closed test outcomes have not been loaded.')


## Changing one input at a time

I keep the settings and training budget fixed and apply the same zero floor to both sets of predictions. There's one wrinkle. Adding a column changes what the tree can draw when it samples features. I ran seed 42 here, so a small difference could look different with another seed. The notebook saves both fits and their feature definitions in a new experiment folder.


In [ ]:
saved_config = json.loads((BUNDLE / 'xgboost_config.json').read_text())
tree_config = saved_config['learner']['gradient_booster']['tree_train_param']
params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse',
          'tree_method': 'hist', 'device': DEVICE}
for name in ['max_depth', 'min_child_weight', 'learning_rate', 'subsample',
             'colsample_bytree', 'reg_alpha', 'reg_lambda', 'max_bin',
             'max_cat_to_onehot', 'max_cat_threshold']:
    params[name] = (int(tree_config[name]) if name in
                   {'max_depth', 'max_bin', 'max_cat_to_onehot', 'max_cat_threshold'}
                   else float(tree_config[name]))
assert params['reg_lambda'] == 10 and params['max_depth'] == 8
display(pd.Series(params, name='shared_parameters'))

def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    assert actual.shape == predicted.shape and np.isfinite(predicted).all()
    error = predicted - actual
    denominator = actual.sum()
    return {'rmse': float(np.sqrt(np.mean(error ** 2))),
            'mae': float(np.mean(np.abs(error))),
            'wape_percent': float(100 * np.abs(error).sum() / denominator) if denominator else np.nan,
            'mean_bias': float(error.mean())}

OUT.mkdir(parents=True, exist_ok=False)
predictions = validation[keys + ['borough', 'total_pickups']].copy()
results = []
for seed in SEEDS:
    for arm, arm_features in [('baseline', features), ('with_borough', features + ['borough'])]:
        started = time.perf_counter()
        train_x = training[arm_features]
        valid_x = validation[arm_features]
        dtrain = xgb.QuantileDMatrix(train_x, label=training.total_pickups,
                                   enable_categorical=True, max_bin=params['max_bin'])
        dvalid = xgb.QuantileDMatrix(valid_x, label=validation.total_pickups,
                                   enable_categorical=True, max_bin=params['max_bin'], ref=dtrain)
        history = {}
        model = xgb.train(dict(params, seed=seed), dtrain, num_boost_round=ROUNDS,
                          evals=[(dvalid, 'validation')], evals_result=history, verbose_eval=100)
        predicted = np.maximum(model.predict(dvalid), 0)
        assert model.num_boosted_rounds() == ROUNDS
        assert model.feature_names == arm_features
        prediction_column = f'{arm}_seed_{seed}'
        predictions[prediction_column] = predicted
        results.append({'arm': arm, 'seed': seed, 'features': len(arm_features),
                        'rounds': ROUNDS, 'training_seconds': time.perf_counter() - started,
                        **metrics(validation.total_pickups, predicted)})
        model_path = OUT / f'{prediction_column}.json'
        model.save_model(model_path)
        reloaded = xgb.Booster()
        reloaded.load_model(model_path)
        np.testing.assert_allclose(np.maximum(reloaded.predict(dvalid), 0), predicted, atol=0.01, rtol=1e-5)
        model_contract = {'ordered_feature_names': arm_features,
                          'xgboost_feature_types': model.feature_types,
                          'categories': {c: list(train_x[c].cat.categories) for c in train_x
                                         if isinstance(train_x[c].dtype, pd.CategoricalDtype)}}
        (OUT / f'{prediction_column}_contract.json').write_text(json.dumps(model_contract, indent=2))
        (OUT / f'{prediction_column}_history.json').write_text(json.dumps(history))
        del model, reloaded, dtrain, dvalid, train_x, valid_x
        gc.collect()
summary = pd.DataFrame(results)
display(summary)


## Did adding borough reduce the errors?

I start with RMSE because it gives more weight to large misses. Then I check MAE, WAPE, and bias to see whether the story changes. I compare models on the same rows at each horizon. One observed zone-hour can appear in several forecasts, so I can't treat those rows as independent evidence.


In [ ]:
paired = summary.pivot(index='seed', columns='arm', values='rmse')
paired['rmse_improvement'] = paired['baseline'] - paired['with_borough']
paired['improvement_percent'] = 100 * paired.rmse_improvement / paired.baseline
display(paired)
group_rows = []
for seed in SEEDS:
    for group in ['forecast_horizon', 'borough', 'location_id']:
        for value, part in predictions.groupby(group, observed=True):
            for arm in ['baseline', 'with_borough']:
                group_rows.append({'seed': seed, 'group': group, 'value': str(value),
                                   'arm': arm, 'rows': len(part),
                                   **metrics(part.total_pickups, part[f'{arm}_seed_{seed}'])})
group_metrics = pd.DataFrame(group_rows)
display(group_metrics.loc[group_metrics.group.isin(['forecast_horizon', 'borough'])])
summary.to_csv(OUT / 'validation_metrics.csv', index=False)
paired.to_csv(OUT / 'paired_rmse_comparison.csv')
group_metrics.to_csv(OUT / 'validation_group_metrics.csv', index=False)
predictions.to_parquet(OUT / 'validation_predictions.parquet', index=False)
lookup.to_csv(OUT / 'zone_borough_lookup.csv', index=False)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

provenance_files = [LOOKUP, BUNDLE / 'xgboost_config.json', BUNDLE / 'feature_contract.json',
                    DATA / 'manifests/production_model_feature_manifest.parquet',
                    DATA / 'manifests/production_forecast_origin_split_manifest.parquet']
run_metadata = {
    'status': 'exploratory_validation_only', 'test_outcomes_loaded': False,
    'seeds': SEEDS, 'rounds': ROUNDS, 'parameters': params,
    'training_rows': len(training), 'validation_rows': len(validation),
    'training_origin_start': str(train_origins.forecast_origin_utc.min()),
    'training_origin_end': str(train_origins.forecast_origin_utc.max()),
    'validation_origin_start': str(valid_origins.forecast_origin_utc.min()),
    'validation_origin_end': str(valid_origins.forecast_origin_utc.max()),
    'geography': '263 zones including EWR; identical scope in both arms',
    'source_hashes': {str(p.relative_to(ROOT)): sha256(p) for p in provenance_files},
    'data_partitions': [{'path': str(p.relative_to(ROOT)), 'bytes': p.stat().st_size,
                         'sha256': sha256(p)} for p in paths
                        if p.parent.parent.name == 'year=2024' or p.parent.parent.name == 'year=2025'
                        or (p.parent.parent.name == 'year=2026' and p.parent.name <= 'month=03')],
    'software': {'python': sys.version, 'xgboost': xgb.__version__, 'pandas': pd.__version__,
                 'platform': platform.platform()},
}
(OUT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2))
print('Saved experiment:', OUT)


## Why I kept the 30-feature model

In my clean run, adding borough raised validation RMSE from **33.30 to 33.47**. I saw a small increase in each of the six horizons. That's enough for me to leave the existing model as it is. To tell whether that result holds beyond this run, I would repeat the paired fit with more seeds and check it on a fresh time window.


## What if I average two different forecasts?

Borough didn't help in this run, so I went back to the original 30 features. This time I ask a different question. If XGBoost overshoots in an hour when LightGBM undershoots, could the two errors partly cancel? I try a 50/50 blend on the same validation rows and compare it with each model on its own.

This section uses the predictions and training data already in memory. LightGBM trains on the CPU, and the full fit can take a while.


In [ ]:
import os
import lightgbm as lgb

required_state = ['training', 'validation', 'predictions', 'features', 'SEEDS',
                  'category_levels', 'metrics', 'OUT', 'RUN_ID']
missing_state = [name for name in required_state if name not in globals()]
if missing_state:
    raise RuntimeError(f'Missing notebook state: {missing_state}. Run the earlier sections first.')

ENSEMBLE_SEED = 42
XGB_WEIGHT = 0.5  # Fixed before evaluating; no weight search on this validation set.
LGB_MAX_ROUNDS = 2000
LGB_EARLY_STOPPING = 100
xgb_prediction_column = f'baseline_seed_{ENSEMBLE_SEED}'
if xgb_prediction_column not in predictions:
    raise RuntimeError(f'Run the baseline for seed {ENSEMBLE_SEED} first.')
assert len(features) == 30 and 'borough' not in features
assert len(predictions) == len(validation)
alignment_keys = ['forecast_origin_utc', 'target_hour_utc', 'location_id', 'forecast_horizon']
pd.testing.assert_frame_equal(predictions[alignment_keys].reset_index(drop=True),
                              validation[alignment_keys].reset_index(drop=True))
np.testing.assert_array_equal(predictions.total_pickups, validation.total_pickups)
assert not predictions.duplicated(alignment_keys).any()
assert np.isfinite(predictions[xgb_prediction_column]).all()
assert predictions[xgb_prediction_column].ge(0).all()

ENSEMBLE_OUT = OUT / ('ensemble_' + uuid.uuid4().hex[:8])
print('LightGBM:', lgb.__version__)
print('Training rows:', f'{len(training):,}')
print('Validation rows:', f'{len(validation):,}')
print('Features:', len(features), '| borough excluded from model inputs')
print('XGBoost weight:', XGB_WEIGHT, '| LightGBM weight:', 1 - XGB_WEIGHT)
print('Alignment checks passed. Training starts in the next cell.')


## Giving LightGBM the same pickup history

I train LightGBM on the same rows as XGBoost. LightGBM uses validation RMSE to decide when to stop, while XGBoost has the fixed round count from the previous comparison. I'm testing whether these two fitted forecasts work well together. Their settings and training rules differ.


In [ ]:
lgb_parameters = {
    'objective': 'regression', 'metric': ['rmse', 'l1'],
    'device_type': 'cpu', 'num_threads': min(14, os.cpu_count() or 1),
    'force_col_wise': True, 'learning_rate': 0.05,
    'num_leaves': 127, 'max_depth': 8, 'min_data_in_leaf': 100,
    'feature_fraction': 0.85, 'bagging_fraction': 0.80, 'bagging_freq': 1,
    'lambda_l1': 0.10, 'lambda_l2': 5.00, 'max_bin': 255,
    'seed': ENSEMBLE_SEED, 'feature_fraction_seed': ENSEMBLE_SEED,
    'bagging_seed': ENSEMBLE_SEED, 'data_random_seed': ENSEMBLE_SEED,
    'deterministic': True, 'first_metric_only': True, 'verbosity': -1,
}
lgb_started = time.perf_counter()
lgb_train_features = training[features]
lgb_valid_features = validation[features]
lgb_train_data = lgb.Dataset(lgb_train_features, label=training.total_pickups,
                             categorical_feature=list(category_levels), free_raw_data=True)
lgb_valid_data = lgb.Dataset(lgb_valid_features, label=validation.total_pickups,
                             categorical_feature=list(category_levels), reference=lgb_train_data,
                             free_raw_data=True)
lgb_history = {}
ensemble_lgb_model = lgb.train(
    params=lgb_parameters, train_set=lgb_train_data, num_boost_round=LGB_MAX_ROUNDS,
    valid_sets=[lgb_valid_data], valid_names=['validation'],
    callbacks=[lgb.early_stopping(LGB_EARLY_STOPPING, first_metric_only=True),
               lgb.log_evaluation(100), lgb.record_evaluation(lgb_history)],
)
assert ensemble_lgb_model.feature_name() == features
lgb_predictions = np.maximum(ensemble_lgb_model.predict(
    lgb_valid_features, num_iteration=ensemble_lgb_model.best_iteration), 0)
assert len(lgb_predictions) == len(validation) and np.isfinite(lgb_predictions).all()
ensemble_lgb_training_seconds = time.perf_counter() - lgb_started
ENSEMBLE_OUT.mkdir(parents=True, exist_ok=False)
lgb_model_path = ENSEMBLE_OUT / 'lightgbm_model.txt'
ensemble_lgb_model.save_model(str(lgb_model_path), num_iteration=ensemble_lgb_model.best_iteration)
# Verify serialization on a small slice without repeating full inference.
lgb_reloaded = lgb.Booster(model_file=str(lgb_model_path))
probe_rows = min(10000, len(validation))
np.testing.assert_allclose(np.maximum(lgb_reloaded.predict(lgb_valid_features.iloc[:probe_rows]), 0),
                           lgb_predictions[:probe_rows], rtol=1e-8, atol=1e-8)
del lgb_train_data, lgb_valid_data, lgb_train_features, lgb_valid_features, lgb_reloaded
gc.collect()
print('Best LightGBM round:', ensemble_lgb_model.best_iteration)
print('Training and full prediction minutes:', round(ensemble_lgb_training_seconds / 60, 2))
print('LightGBM validation metrics:', metrics(validation.total_pickups, lgb_predictions))


## Do their mistakes cancel out?

I line up forecasts by origin, horizon, and zone before averaging. Then I compare the blend with the better standalone model. I also check whether the errors move together. If both models are usually too high or too low at the same times, averaging can only do so much.


In [ ]:
ensemble_evaluation = predictions[alignment_keys + ['borough', 'total_pickups']].copy()
ensemble_evaluation['xgboost'] = predictions[xgb_prediction_column].to_numpy()
ensemble_evaluation['lightgbm'] = lgb_predictions
ensemble_evaluation['ensemble_50_50'] = (
    XGB_WEIGHT * ensemble_evaluation.xgboost
    + (1 - XGB_WEIGHT) * ensemble_evaluation.lightgbm
)
ensemble_names = ['xgboost', 'lightgbm', 'ensemble_50_50']
ensemble_comparison = pd.DataFrame([
    {'model': name, **metrics(ensemble_evaluation.total_pickups, ensemble_evaluation[name])}
    for name in ensemble_names
])
display(ensemble_comparison)
standalone_rmse = ensemble_comparison.set_index('model').loc[['xgboost', 'lightgbm'], 'rmse'].min()
blend_rmse = ensemble_comparison.set_index('model').loc['ensemble_50_50', 'rmse']
print('Ensemble RMSE improvement over better standalone:', round(standalone_rmse - blend_rmse, 6))
xgb_residual = ensemble_evaluation.xgboost.to_numpy() - ensemble_evaluation.total_pickups.to_numpy()
lgb_residual = ensemble_evaluation.lightgbm.to_numpy() - ensemble_evaluation.total_pickups.to_numpy()
# Elementwise sums avoid a BLAS dependency for this two-vector statistic.
xgb_centered = xgb_residual - xgb_residual.mean()
lgb_centered = lgb_residual - lgb_residual.mean()
correlation_denominator = np.sqrt(np.sum(xgb_centered ** 2) * np.sum(lgb_centered ** 2))
residual_correlation = (float(np.sum(xgb_centered * lgb_centered) / correlation_denominator)
                        if correlation_denominator > 0 else None)
print('Validation residual correlation:', residual_correlation)
ensemble_group_records = []
for group in ['forecast_horizon', 'borough']:
    for value, part in ensemble_evaluation.groupby(group, observed=True):
        for name in ensemble_names:
            ensemble_group_records.append({'group': group, 'value': str(value), 'model': name,
                                            'rows': len(part), **metrics(part.total_pickups, part[name])})
ensemble_group_metrics = pd.DataFrame(ensemble_group_records)
display(ensemble_group_metrics)
ensemble_comparison.to_csv(ENSEMBLE_OUT / 'validation_metrics.csv', index=False)
ensemble_group_metrics.to_csv(ENSEMBLE_OUT / 'validation_group_metrics.csv', index=False)
ensemble_evaluation.to_parquet(ENSEMBLE_OUT / 'validation_predictions.parquet', index=False)
xgb_model_path = OUT / f'{xgb_prediction_column}.json'
assert xgb_model_path.is_file(), 'Baseline artifact must belong to this same experiment run.'
ensemble_contract = {
    'status': 'exploratory_validation_only', 'test_outcomes_loaded': False,
    'parent_run_id': RUN_ID, 'seed': ENSEMBLE_SEED,
    'ordered_feature_names': features,
    'categories': {name: list(training[name].cat.categories) for name in category_levels},
    'xgboost_model': str(xgb_model_path.relative_to(ROOT)),
    'xgboost_sha256': sha256(xgb_model_path),
    'xgboost_weight': XGB_WEIGHT,
    'lightgbm_model': str(lgb_model_path.relative_to(ROOT)),
    'lightgbm_sha256': sha256(lgb_model_path),
    'lightgbm_weight': 1 - XGB_WEIGHT,
    'prediction_rule': 'Clip each model prediction at zero before taking the weighted average.',
    'lightgbm_parameters': lgb_parameters,
    'lightgbm_best_iteration': ensemble_lgb_model.best_iteration,
    'lightgbm_max_rounds': LGB_MAX_ROUNDS, 'early_stopping_rounds': LGB_EARLY_STOPPING,
    'lightgbm_training_and_prediction_seconds': ensemble_lgb_training_seconds,
    'lightgbm_version': lgb.__version__, 'residual_correlation': residual_correlation,
    'validation_rows': len(validation),
    'validation_origin_start': str(validation.forecast_origin_utc.min()),
    'validation_origin_end': str(validation.forecast_origin_utc.max()),
}
(ENSEMBLE_OUT / 'ensemble_contract.json').write_text(json.dumps(ensemble_contract, indent=2))
(ENSEMBLE_OUT / 'lightgbm_history.json').write_text(json.dumps(lgb_history, indent=2))
print('Saved ensemble experiment:', ENSEMBLE_OUT)


## What I would keep from this experiment

The blend scored **32.94 RMSE**, compared with **32.98** for LightGBM and **33.30** for XGBoost. That's about **0.04 pickups of RMSE** better than LightGBM alone. Their errors were highly correlated at **0.976**, which helps explain why averaging only made a small difference.

I'd keep the blend as an experiment. Running two models adds inference work for a small gain on a validation period I already used during development. Before using the blend in the replay, I'd want to see whether the improvement holds on a fresh period.
